# RT-LCOD — Notebook nghiên cứu từ đầu đến cuối

Kiến trúc được hiện thực trong nhánh này:

**YOLOE-26M → candidate mở → đặc trưng vùng nhẹ → head grounding thuộc tính/quan hệ → mục tiêu hoặc NO_TARGET**, trong đó **Grounding DINO chỉ đóng vai trò teacher ngoại tuyến**.

Notebook mặc định chạy an toàn. Với `RUN_HEAVY = False`, **Run All** không tải YOLOE, Grounding DINO hoặc bộ dữ liệu bên ngoài. Toàn bộ pipeline cục bộ — parser, hình học, forward/backward của student, loss, checkpoint, đo thời gian, metric và tracking — được kiểm tra bằng dữ liệu tổng hợp có tính xác định.


## Phiên 0 — Thiết lập Colab, chế độ chạy và thư mục project

Cell này tự tìm hoặc clone đúng nhánh project, cài dependency vào runtime Colab, thêm `src/` vào `sys.path` và đặt cờ cho các bước nặng. Giữ `RUN_HEAVY=False` để **Run All** luôn chạy hết smoke pipeline mà không cần dataset hay checkpoint ngoài.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/dangtai111325/VLM.git"
BRANCH = "feature/rt-lcod-yoloe-student"
HERE = Path.cwd().resolve()


def find_project_root() -> Path | None:
    candidates = [
        HERE,
        HERE / "rt_lcod",
        HERE / "training_test_model" / "rt_lcod",
        HERE.parent / "rt_lcod",
        HERE.parent / "training_test_model" / "rt_lcod",
    ]
    return next(
        (candidate for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )


ROOT = find_project_root()
if ROOT is None:
    checkout_dir = Path("/content/training_test_model")
    if not checkout_dir.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                REPOSITORY_URL,
                str(checkout_dir),
            ]
        )
    ROOT = checkout_dir / "rt_lcod"

if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Không tìm thấy project RT-LCOD tại: {ROOT}")

INSTALL_HEAVY = True  # Cài dependency YOLOE/teacher cho UI Colab; model chỉ tải khi bạn bắt đầu stream.
install_target = f"{ROOT}[all,ui]" if INSTALL_HEAVY else f"{ROOT}[notebook,dev,ui]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target])

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RUN_HEAVY = False
print("Thư mục RT-LCOD:", ROOT)
print("RUN_HEAVY:", RUN_HEAVY)


## Phiên 1 — Kiểm tra nhanh môi trường

Trước tiên, notebook ghi nhận trạng thái Python/PyTorch/CUDA. Việc thiếu package cho mô hình nặng chỉ được thông báo, không gây lỗi khi chạy smoke mode.


In [ ]:
import importlib.util, platform, torch
from rt_lcod.utils.logging import environment_snapshot

env = environment_snapshot()
print(env)
for package in ["torch", "torchvision", "yaml", "ultralytics", "transformers", "cv2"]:
    print(f"{package:12s}", "OK" if importlib.util.find_spec(package) else "missing/optional")


## Phiên 2 — Nạp cấu hình và seed

`smoke.yaml` nhỏ, chạy được trên CPU và dùng để kiểm tra toàn bộ luồng. Thí nghiệm thật dùng `base.yaml`. Mọi thí nghiệm có thể lặp lại đều bắt đầu bằng việc cố định seed.


In [ ]:
from rt_lcod.config import load_config
from rt_lcod.utils.seed import seed_everything

CONFIG_PATH = ROOT / "configs" / ("base.yaml" if RUN_HEAVY else "smoke.yaml")
config = load_config(CONFIG_PATH)
seed_everything(config.seed, deterministic=not RUN_HEAVY)
print(config)


## Phiên 3 — Hợp đồng phân tích prompt

V1 hỗ trợ một lớp mục tiêu, tối đa một thuộc tính, và tối đa một quan hệ bậc một với vật tham chiếu. Parser được tách khỏi grounding để có thể đo lỗi parser một cách độc lập.


In [ ]:
from rt_lcod.prompt import RuleSlotParser

parser = RuleSlotParser()
examples = [
    "cup",
    "red cup",
    "red cup next to pillow",
    "bottle left of laptop",
]
for text in examples:
    print(text, "->", parser.parse(text))


## Phiên 4 — Đặc trưng hình học cho quan hệ một bước

Relation head không cần một transformer đa phương thức lớn. Nó nhận đặc trưng thị giác của candidate và hình học chuẩn hoá theo từng cặp. Nhờ vậy các quan hệ trái/phải/trên/dưới/gần/chồng lấp có thể học với chi phí thấp mỗi frame.


In [ ]:
import torch
from rt_lcod.geometry import relation_geometry

target = torch.tensor([[0.10, 0.20, 0.30, 0.40]])
reference = torch.tensor([[0.60, 0.20, 0.80, 0.40]])
features = relation_geometry(target, reference)
print("Shape đặc trưng hình học:", tuple(features.shape))
print("dx, dy, khoảng cách:", features[0, :3].tolist())


## Phiên 5 — Dữ liệu candidate tổng hợp

Trước khi dùng mô hình nặng, ta kiểm tra hợp đồng đầu vào của student bằng tensor candidate có cùng shape với đầu ra YOLOE đã cache. Cách này phát hiện sớm và rẻ các lỗi interface, padding, loss và checkpoint.


In [ ]:
from torch.utils.data import DataLoader
from rt_lcod.data.synthetic import SyntheticGroundingDataset
from rt_lcod.data.cached_dataset import collate_cached

dataset = SyntheticGroundingDataset(
    size=16,
    visual_dim=config.model.visual_dim,
    text_dim=config.model.text_dim,
    max_candidates=min(config.model.top_k, 6),
    seed=config.seed,
)
loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_cached)
batch = next(iter(loader))
for key, value in batch.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape), value.dtype)


## Phiên 6 — Forward pass của student

Student tính điểm cho `N candidate + NO_TARGET`. Candidate thuộc lớp mục tiêu được lọc theo nhãn YOLOE; thành phần thuộc tính và quan hệ sau đó phân biệt các candidate cùng lớp còn mơ hồ.


In [ ]:
from rt_lcod.models.student import RTLCODStudent

student = RTLCODStudent(config.model)
output = student(batch)
print("Logit candidate:", tuple(output.candidate_logits.shape))
print("Logit đầy đủ (gồm NO_TARGET):", tuple(output.logits.shape))
print("Dự đoán:", output.logits.argmax(dim=-1).tolist())


## Phiên 7 — Loss và một bước tối ưu

Mục tiêu huấn luyện kết hợp chọn mục tiêu, giám sát thuộc tính tuỳ chọn, giám sát cặp quan hệ đã xác minh, giám sát `NO_TARGET` tường minh, và distillation từ teacher khi có.


In [ ]:
from rt_lcod.training.losses import compute_loss

optimizer = torch.optim.AdamW(student.parameters(), lr=config.training.learning_rate)
losses = compute_loss(output, batch, config.loss, config.training.kd_temperature)
optimizer.zero_grad(set_to_none=True)
losses.total.backward()
optimizer.step()
print(losses.scalar_dict())


## Phiên 8 — Dashboard theo dõi huấn luyện

Dashboard cập nhật theo batch/epoch: stage, tiến độ tổng và tiến độ phase hiện tại, loss, metric, learning rate, global step, GPU/VRAM, thời gian đã chạy và ETA. Các giá trị này lấy trực tiếp từ trainer, không phải số giả lập.


In [ ]:
from html import escape
import math
import time

import ipywidgets as widgets
from IPython.display import HTML, display


def format_duration(seconds: float | None) -> str:
    if seconds is None or not math.isfinite(seconds):
        return "Đang ước tính"
    seconds = max(0, round(seconds))
    hours, remainder = divmod(seconds, 3_600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class TrainingMonitor:
    EVENT_LABELS = {
        "started": "Đang khởi tạo huấn luyện",
        "epoch_started": "Đang chạy phase",
        "batch_completed": "Đang xử lý batch",
        "epoch_phase_completed": "Đã hoàn thành phase",
        "epoch_completed": "Đã hoàn thành epoch",
        "completed": "Huấn luyện hoàn tất",
        "resumed": "Đã tiếp tục từ checkpoint",
    }

    def __init__(self) -> None:
        self.started_at = time.perf_counter()
        self.total_units = 1
        self.train_batches = 0
        self.val_batches = 0
        self.phase_progress = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            description="Phase",
            bar_style="info",
            layout=widgets.Layout(width="100%"),
        )
        self.total_progress = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            description="Toàn bộ",
            bar_style="success",
            layout=widgets.Layout(width="100%"),
        )
        self.status = widgets.HTML()
        self.summary = widgets.HTML()
        self.metrics = widgets.HTML()
        self.timing_details = widgets.HTML()
        self.view = widgets.VBox(
            [
                self.status,
                self.total_progress,
                self.phase_progress,
                self.summary,
                self.metrics,
                self.timing_details,
            ],
            layout=widgets.Layout(
                width="100%",
                border="1px solid #475569",
                padding="16px",
                margin="8px 0",
            ),
        )

    def show(self) -> None:
        display(
            HTML(
                "<style>.widget-label { min-width: 80px; } "
                ".jupyter-widgets { font-variant-numeric: tabular-nums; }</style>"
            )
        )
        display(self.view)

    @staticmethod
    def _metric(value: float | None, digits: int = 4) -> str:
        if value is None or not math.isfinite(value):
            return "—"
        return f"{value:.{digits}f}"

    def __call__(self, payload: dict) -> None:
        event = payload["event"]
        total_epochs = int(payload.get("total_epochs", config.training.epochs))
        if event == "started":
            self.train_batches = int(payload.get("train_batches") or 0)
            self.val_batches = int(payload.get("val_batches") or 0)
            self.total_units = max(total_epochs * max(self.train_batches + self.val_batches, 1), 1)
            self.total_progress.max = self.total_units

        epoch = int(payload.get("epoch", 0))
        mode = payload.get("mode")
        batch_index = int(payload.get("batch_index") or 0)
        total_batches = int(payload.get("total_batches") or 1)
        if event == "batch_completed":
            phase_offset = 0 if mode == "train" else self.train_batches
            completed_units = epoch * (self.train_batches + total_batches) + phase_offset + batch_index
            self.total_progress.value = min(completed_units, self.total_progress.max)
            self.phase_progress.max = max(total_batches, 1)
            self.phase_progress.value = min(batch_index, self.phase_progress.max)
            self.phase_progress.description = f"{mode or 'phase'} · epoch {epoch + 1}/{total_epochs}"
        elif event == "epoch_completed":
            self.total_progress.value = min((epoch + 1) * (self.train_batches + self.val_batches), self.total_progress.max)

        elapsed_s = time.perf_counter() - self.started_at
        fraction = self.total_progress.value / max(self.total_progress.max, 1)
        eta_s = elapsed_s * (1.0 - fraction) / fraction if fraction > 0 else None
        label = self.EVENT_LABELS.get(event, event)
        device = escape(str(payload.get("device", "—")))
        self.status.value = (
            f"<div role='status' aria-live='polite' style='color:#f8fafc; background:#1e293b; "
            f"padding:12px 16px; border-radius:8px; font-weight:600'>{escape(label)}</div>"
        )
        self.summary.value = (
            "<div style='display:flex; flex-wrap:wrap; gap:16px; color:#334155; line-height:1.6'>"
            f"<span><b>Thiết bị:</b> {device}</span>"
            f"<span><b>Epoch:</b> {epoch + 1}/{total_epochs}</span>"
            f"<span><b>Global step:</b> {payload.get('global_step', 0):,}</span>"
            f"<span><b>Đã chạy:</b> {format_duration(elapsed_s)}</span>"
            f"<span><b>ETA:</b> {format_duration(eta_s)}</span>"
            f"<span><b>VRAM đang dùng:</b> {payload.get('gpu_memory_allocated_mib', 0.0):.0f} MiB</span>"
            f"<span><b>VRAM dự trữ:</b> {payload.get('gpu_memory_reserved_mib', 0.0):.0f} MiB</span>"
            f"<span><b>VRAM đỉnh:</b> {payload.get('gpu_memory_peak_mib', 0.0):.0f} MiB</span>"
            "</div>"
        )
        metrics = payload.get("metrics") or payload.get("val_metrics") or {}
        self.metrics.value = (
            "<div style='display:flex; flex-wrap:wrap; gap:16px; color:#0f172a; margin-top:8px'>"
            f"<span><b>Loss:</b> {self._metric(metrics.get('loss_total'))}</span>"
            f"<span><b>Top-1:</b> {self._metric(metrics.get('target_top1_accuracy'))}</span>"
            f"<span><b>No-target:</b> {self._metric(metrics.get('no_target_accuracy'))}</span>"
            f"<span><b>LR:</b> {self._metric(payload.get('lr'), 7)}</span>"
            f"<span><b>Best score:</b> {self._metric(payload.get('best_metric'))}</span>"
            "</div>"
        )
        timings = payload.get("timings") or {}
        if timings:
            timing_rows = "".join(
                f"<span><b>{escape(name)}:</b> {values['mean_ms']:.1f} ms</span>"
                for name, values in timings.items()
            )
            self.timing_details.value = (
                "<div style='margin-top:8px; color:#334155'><b>Thời gian trung bình:</b> "
                f"{timing_rows}</div>"
            )
        if event == "completed":
            self.total_progress.bar_style = "success"
            self.phase_progress.bar_style = "success"


training_monitor = TrainingMonitor()
training_monitor.show()


## Phiên 9 — Huấn luyện smoke hoàn chỉnh, checkpoint và theo dõi run

Cell này chạy đủ hai epoch synthetic để kiểm tra luồng train/validation/checkpoint thực tế. Dashboard phía trên cập nhật theo batch và epoch. Mỗi run có thể tiếp tục từ checkpoint, vốn lưu model, optimizer, scheduler, scaler, epoch/global step, best metric và trạng thái RNG. Thư mục run cũng lưu môi trường, config, metric và thời gian.


In [ ]:
from rt_lcod.training.checkpoint import load_checkpoint
from rt_lcod.training.trainer import Trainer
from rt_lcod.utils.logging import RunLogger

train_dataset = SyntheticGroundingDataset(
    size=16,
    visual_dim=config.model.visual_dim,
    text_dim=config.model.text_dim,
    max_candidates=min(config.model.top_k, 6),
    seed=config.seed,
)
validation_dataset = SyntheticGroundingDataset(
    size=8,
    visual_dim=config.model.visual_dim,
    text_dim=config.model.text_dim,
    max_candidates=min(config.model.top_k, 6),
    seed=config.seed + 1_000,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    collate_fn=collate_cached,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    collate_fn=collate_cached,
)

run_logger = RunLogger(ROOT / "artifacts" / "notebook_smoke_runs", "notebook_smoke")
trainer = Trainer(
    RTLCODStudent(config.model),
    config,
    run_logger,
    progress_callback=training_monitor,
)
best_checkpoint = trainer.fit(train_loader, validation_loader)

restored = RTLCODStudent(config.model)
state = load_checkpoint(best_checkpoint, model=restored, restore_rng=False)
print("Thư mục run:", run_logger.run_dir)
print("Checkpoint tốt nhất:", best_checkpoint)
print("Checkpoint đã khôi phục:", {key: state[key] for key in ["epoch", "global_step", "best_metric"]})


## Phiên 10 — Kiểm tra tracker theo thời gian

Tracking là tối ưu cho triển khai, không thay thế việc đánh giá LCOD. Cần báo cáo riêng FPS thô của model và FPS thực tế của hệ có tracker.


In [ ]:
from rt_lcod.inference.temporal import SimpleTargetTracker

tracker = SimpleTargetTracker(iou_gate=0.2, max_missed=2)
print(tracker.update([((10, 10, 30, 30), 0.90)]))
print(tracker.update([((12, 11, 32, 31), 0.85)]))
print(tracker.update([]))


## Phiên 11 — Chạy bộ test tự động

Sau smoke training, cell này chạy toàn bộ unit test. Nếu cell hoàn tất, mọi kiểm tra parser, hình học, forward/backward, checkpoint và tracker đều đã pass.


In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode:
    print(test_result.stderr)
    raise RuntimeError("Unit test thất bại; xem log ở trên để biết chi tiết.")
print("Toàn bộ unit test: PASS")


## Phiên 12 — Chuẩn bị Stage 0/1 nặng (có cờ bảo vệ)

Khi `RUN_HEAVY=True`, các lệnh sau là luồng chạy thật:

1. kiểm tra manifest chuẩn;
2. chạy YOLOE-26M để cache candidate mục tiêu/tham chiếu;
3. tính đặc trưng vùng MobileNetV3 đã đóng băng;
4. cache embedding văn bản;
5. huấn luyện grounding head.

Notebook chỉ in lệnh, không tự âm thầm tải model hoặc giả định dữ liệu thật đã tồn tại. Vì vậy, Run All ở smoke mode luôn kết thúc an toàn.


In [ ]:
commands = [
    "python scripts/validate_manifest.py --manifest data/manifests/train.jsonl",
    "python scripts/cache_candidates.py --config configs/base.yaml --manifest data/manifests/train.jsonl --output data/candidate_cache/train --model yoloe-26m.pt",
    "python scripts/cache_candidates.py --config configs/base.yaml --manifest data/manifests/val.jsonl --output data/candidate_cache/val --model yoloe-26m.pt",
    "python scripts/train.py --config configs/base.yaml --train-cache data/candidate_cache/train --val-cache data/candidate_cache/val --run-name stage1",
]
print("Chế độ mô hình nặng:", RUN_HEAVY)
for command in commands:
    print("  ", command)


## Phiên 13 — Distillation từ teacher (có cờ bảo vệ)

Grounding DINO chỉ là **teacher ngoại tuyến**. Dự đoán của teacher được cache một lần, căn chỉnh với candidate của student rồi đổi thành phân phối trên candidate cộng `NO_TARGET`. Inference của student không cần teacher.


In [ ]:
teacher_commands = [
    "python scripts/cache_teacher.py --manifest data/manifests/train.jsonl --candidate-cache data/candidate_cache/train --output data/teacher_cache/train",
    "python scripts/train.py --config configs/base.yaml --train-cache data/candidate_cache/train --val-cache data/candidate_cache/val --teacher-cache data/teacher_cache/train --resume runs/<stage1>/checkpoints/best.pt --run-name stage2_kd",
]
for command in teacher_commands:
    print("  ", command)


## Phiên 14 — Đánh giá và benchmark thời gian thực

Không được kết luận model thời gian thực chỉ từ FPS trung bình. Hãy ghi nhận độ trễ mean/p50/p95/p99, tỷ lệ trễ deadline, VRAM và độ trễ đổi prompt. Đánh giá riêng các nhóm: chỉ lớp, thuộc tính, quan hệ, kết hợp và không có mục tiêu.


In [ ]:
print("Đánh giá:")
print("  python scripts/evaluate.py --config configs/base.yaml --cache data/candidate_cache/val --checkpoint runs/<run>/checkpoints/best.pt")
print("Đo thời gian chạy thực tế:")
print('  python scripts/benchmark_runtime.py --config configs/base.yaml --checkpoint runs/<run>/checkpoints/best.pt --video ../test.mp4 --prompt "red cup next to pillow"')


## Phiên 15 — Điều kiện qua từng stage

Trước khi chuyển stage:

- **Proposal gate:** YOLOE phải thực sự đề xuất được vật mục tiêu/tham chiếu.
- **Grounding gate:** head phải tốt hơn cách chọn candidate có confidence detector cao nhất ở cảnh mơ hồ.
- **Negative gate:** phải đo false positive của `NO_TARGET`.
- **KD gate:** distillation phải tăng độ chính xác mà không tăng chi phí runtime.
- **Realtime gate:** phải đo độ trễ p95 trên GPU robot mục tiêu.
- **Robot gate:** phải tách độ đúng của perception khỏi độ đúng của grasp/manipulation.

Lý do thiết kế, dữ liệu, loss, chiến lược hard-negative, các stage huấn luyện và kế hoạch triển khai robot đầy đủ nằm trong `../../plan.md`.


## Phiên 16 — UI stream video và prompt

UI cuối cùng dùng `test.mp4` nếu file có trong thư mục gốc project, hoặc cho phép upload video khác. Nhập prompt, chọn engine, bấm **Bắt đầu stream** để xem frame đã xử lý, box, track ID, confidence, FPS và latency cập nhật liên tục. Nút **Dừng stream** huỷ generator ngay.

- **Chỉ phát video:** xác nhận video/UI hoạt động, không cần model nặng.
- **YOLOE open-vocabulary:** baseline theo lớp mục tiêu trong prompt; yêu cầu `INSTALL_HEAVY=True`.
- **LCOD checkpoint:** dùng checkpoint LCOD thật được train từ candidate cache với `base.yaml`. Checkpoint smoke phía trên chỉ để kiểm tra pipeline, không dùng để detect video thật.


In [ ]:
from collections import deque
from pathlib import Path
import time

import cv2
import gradio as gr
import numpy as np
import torch

from rt_lcod.config import load_config
from rt_lcod.inference.runtime import RTLCODRuntime
from rt_lcod.inference.temporal import SimpleTargetTracker
from rt_lcod.models.yoloe_adapter import YOLOECandidateGenerator
from rt_lcod.prompt import RuleSlotParser


DEFAULT_VIDEO = ROOT.parent / "test.mp4"
DEFAULT_CHECKPOINT = str(best_checkpoint) if "best_checkpoint" in globals() else ""


def draw_stream_overlay(frame_bgr, box, confidence, track_id, status):
    canvas = frame_bgr.copy()
    color = (34, 197, 94) if box is not None else (148, 163, 184)
    if box is not None:
        x1, y1, x2, y2 = (int(value) for value in box)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        label = f"target | id={track_id} | {confidence:.2f}"
        cv2.putText(canvas, label, (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    cv2.rectangle(canvas, (0, 0), (min(canvas.shape[1], 760), 34), (15, 23, 42), -1)
    cv2.putText(canvas, status, (12, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (248, 250, 252), 1)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


def prepare_stream_engine(engine_name, prompt, checkpoint_path, confidence_threshold):
    parsed_prompt = RuleSlotParser().parse(prompt)
    config_path = ROOT / "configs" / "base.yaml"
    if engine_name == "LCOD checkpoint":
        checkpoint = Path(checkpoint_path).expanduser()
        if not checkpoint.is_file():
            raise ValueError("Chọn đường dẫn checkpoint LCOD thật trước khi bắt đầu stream.")
        runtime = RTLCODRuntime(load_config(config_path), str(checkpoint))
        runtime.set_prompt(prompt)
        return parsed_prompt, runtime
    if engine_name == "YOLOE open-vocabulary":
        config = load_config(config_path)
        device = config.device if torch.cuda.is_available() else "cpu"
        detector = YOLOECandidateGenerator(
            config.model.detector_name,
            device=device,
            imgsz=int(config.runtime["input_size"]),
        )
        return parsed_prompt, detector
    return parsed_prompt, None


def stream_video(video_path, prompt, engine_name, checkpoint_path, confidence_threshold, display_fps):
    if not video_path:
        yield None, "**Lỗi:** Chọn hoặc upload video trước khi bắt đầu."
        return
    try:
        parsed_prompt, engine = prepare_stream_engine(
            engine_name,
            prompt,
            checkpoint_path,
            confidence_threshold,
        )
    except Exception as exc:
        yield None, f"**Không thể khởi tạo engine:** `{exc}`"
        return

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        yield None, f"**Lỗi:** Không mở được video `{video_path}`."
        return

    tracker = SimpleTargetTracker(iou_gate=0.25, max_missed=5)
    latencies_ms = deque(maxlen=120)
    source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
    frame_index = 0
    try:
        while True:
            loop_started = time.perf_counter()
            ok, frame_bgr = capture.read()
            if not ok:
                break
            frame_index += 1
            box = None
            confidence = 0.0
            track_id = None
            if engine_name == "LCOD checkpoint":
                result = engine.process_frame(frame_bgr)
                track = tracker.update(
                    [(result.box, result.confidence)] if result.box is not None else []
                )
                if track is not None:
                    box, confidence, track_id = track.box, track.score, track.track_id
            elif engine_name == "YOLOE open-vocabulary":
                classes = [parsed_prompt.target_class]
                if parsed_prompt.reference_class:
                    classes.append(parsed_prompt.reference_class)
                detections = engine.predict(frame_bgr, classes, conf=float(confidence_threshold))
                target_detections = [
                    (detection.box, detection.score)
                    for detection in detections
                    if detection.label.lower().strip() == parsed_prompt.target_class.lower()
                ]
                track = tracker.update(target_detections)
                if track is not None:
                    box, confidence, track_id = track.box, track.score, track.track_id

            latency_ms = (time.perf_counter() - loop_started) * 1_000
            latencies_ms.append(latency_ms)
            mean_ms = float(np.mean(latencies_ms))
            p95_ms = float(np.percentile(latencies_ms, 95))
            inference_fps = 1_000.0 / max(mean_ms, 1e-6)
            status = f"{engine_name} | frame {frame_index} | {inference_fps:.1f} FPS"
            annotated = draw_stream_overlay(frame_bgr, box, confidence, track_id, status)
            target_state = "đang bám mục tiêu" if box is not None else "không có mục tiêu"
            details = (
                f"**Trạng thái:** {target_state}  \n"
                f"**Prompt:** `{prompt}`  \n"
                f"**Frame:** {frame_index} · **Nguồn:** {source_fps:.1f} FPS · "
                f"**Suy luận:** {inference_fps:.1f} FPS · **Latency:** {latency_ms:.1f} ms · p95 {p95_ms:.1f} ms"
            )
            yield annotated, details
            wait_s = 1.0 / max(float(display_fps), 1.0) - (time.perf_counter() - loop_started)
            if wait_s > 0:
                time.sleep(wait_s)
    finally:
        capture.release()


try:
    video_ui.close()
except NameError:
    pass

ui_css = """
#rtlcod-ui { max-width: 1440px; margin: 0 auto; }
#rtlcod-ui .panel { border: 1px solid #475569; border-radius: 12px; }
#rtlcod-ui .status-panel { background: #0f172a; color: #f8fafc; border-radius: 10px; padding: 12px; }
"""

with gr.Blocks(css=ui_css, title="RT-LCOD Stream Console", elem_id="rtlcod-ui") as video_ui:
    gr.Markdown("# RT-LCOD Stream Console\nNhập prompt, chọn engine rồi bấm **Bắt đầu stream**.")
    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            source_video = gr.File(
                value=str(DEFAULT_VIDEO) if DEFAULT_VIDEO.is_file() else None,
                file_types=["video"],
                type="filepath",
                label="Video nguồn",
            )
            prompt_input = gr.Textbox(
                value="red cup next to pillow",
                label="Prompt LCOD",
                info="Ví dụ: cup, red cup, red cup next to pillow.",
            )
            engine_input = gr.Radio(
                choices=["Chỉ phát video", "YOLOE open-vocabulary", "LCOD checkpoint"],
                value="YOLOE open-vocabulary",
                label="Engine",
            )
            checkpoint_input = gr.Textbox(
                value=DEFAULT_CHECKPOINT,
                label="Checkpoint LCOD",
                info="Chỉ dùng cho checkpoint thật train với base.yaml.",
            )
            confidence_input = gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Ngưỡng confidence")
            display_fps_input = gr.Slider(1, 30, value=15, step=1, label="FPS hiển thị tối đa")
            with gr.Row():
                start_button = gr.Button("Bắt đầu stream", variant="primary")
                stop_button = gr.Button("Dừng stream", variant="stop")
        with gr.Column(scale=2):
            stream_frame = gr.Image(label="Luồng video đã xử lý", streaming=True, format="jpeg")
            stream_status = gr.Markdown("**Trạng thái:** Sẵn sàng.", elem_classes=["status-panel"])

    stream_event = start_button.click(
        fn=stream_video,
        inputs=[
            source_video,
            prompt_input,
            engine_input,
            checkpoint_input,
            confidence_input,
            display_fps_input,
        ],
        outputs=[stream_frame, stream_status],
        show_progress="minimal",
        concurrency_limit=1,
    )
    stop_button.click(fn=None, cancels=[stream_event], queue=False)

video_ui.queue(default_concurrency_limit=1).launch(share=False, prevent_thread_lock=True)
